In [1]:
import requests

BASE_URL = "https://fakestoreapi.com"

In [2]:
class User:
    def __init__(self, user_id, name, email):
        self.user_id = user_id
        self.name    = name
        self.email   = email

    def __str__(self):
        return f"{self.name} ({self.email})"

In [3]:
class Product:
    def __init__(self, product_id, title, price):
        self.product_id = product_id
        self.title      = title
        self.price      = price

    def __str__(self):
        return f"  • {self.title} - ₹{self.price}"


In [4]:
class Cart:
    def __init__(self, cart_id, user, products):
        self.cart_id  = cart_id
        self.user     = user
        self.products = products

    def total(self):
        return round(sum(p.price for p in self.products), 2)

    def display(self):
        print(f"\nCart ID : {self.cart_id}")
        print(f"User    : {self.user or 'Unknown'}")
        print("Products:")
        for p in self.products:
            print(p)
        print(f"Total   : ₹{self.total()}")


In [5]:
def fetch(endpoint):
    """One place to do all API calls — easy to change the URL later."""
    response = requests.get(BASE_URL + "/" + endpoint)
    response.raise_for_status()          # crash early on bad status
    return response.json()

In [6]:
def get_users():
    """Returns {user_id: User} — dict makes lookup O(1) instead of a loop."""
    users = {}
    for u in fetch("users"):
        users[u["id"]] = User(
            user_id = u["id"],
            name    = u["name"]["firstname"] + " " + u["name"]["lastname"],
            email   = u["email"],
        )
    return users

In [7]:
def get_products():
    """Returns {product_id: Product}."""
    products = {}
    for p in fetch("products"):
        products[p["id"]] = Product(
            product_id = p["id"],
            title      = p["title"],
            price      = p["price"],
        )
    return products


In [8]:
def get_carts(users, products):
    carts = []
    for c in fetch("carts"):
        cart_products = [
            products[item["productId"]]
            for item in c["products"]
            if item["productId"] in products      # skip missing products safely
        ]
        carts.append(Cart(
            cart_id  = c["id"],
            user     = users.get(c["userId"]),
            products = cart_products,
        ))
    return carts

In [9]:
def main():
    users    = get_users()
    products = get_products()
    carts    = get_carts(users, products)

    for cart in carts:
        cart.display()


main()


Cart ID : 1
User    : john doe (john@gmail.com)
Products:
  • Fjallraven - Foldsack No. 1 Backpack, Fits 15 Laptops - ₹109.95
  • Mens Casual Premium Slim Fit T-Shirts  - ₹22.3
  • Mens Cotton Jacket - ₹55.99
Total   : ₹188.24

Cart ID : 2
User    : john doe (john@gmail.com)
Products:
  • Mens Casual Premium Slim Fit T-Shirts  - ₹22.3
  • Fjallraven - Foldsack No. 1 Backpack, Fits 15 Laptops - ₹109.95
  • John Hardy Women's Legends Naga Gold & Silver Dragon Station Chain Bracelet - ₹695
Total   : ₹827.25

Cart ID : 3
User    : david morrison (morrison@gmail.com)
Products:
  • Fjallraven - Foldsack No. 1 Backpack, Fits 15 Laptops - ₹109.95
  • WD 2TB Elements Portable External Hard Drive - USB 3.0  - ₹64
Total   : ₹173.95

Cart ID : 4
User    : kevin ryan (kevin@gmail.com)
Products:
  • Fjallraven - Foldsack No. 1 Backpack, Fits 15 Laptops - ₹109.95
Total   : ₹109.95

Cart ID : 5
User    : kevin ryan (kevin@gmail.com)
Products:
  • White Gold Plated Princess - ₹9.99
  • Pierced Owl Ros